In [1]:
import json
from pathlib import Path 

wandb_dir = Path("wandb_downloads_scorers")
input_file = wandb_dir / "scorers-ho-v5" / "results.json"

with input_file.open("r") as f:
    data = json.load(f)

In [4]:
data.pop("null")

[{'module_name': '/tmp/tmp0g6wa0cw',
  'config': {'_attn_implementation_autoset': {'value': True},
   '_name_or_path': {'value': 'prajjwal1/bert-tiny'},
   'accelerator_config': {'value': {'dispatch_batches': None,
     'even_batches': True,
     'gradient_accumulation_kwargs': None,
     'non_blocking': False,
     'split_batches': False,
     'use_seedable_sampler': True}},
   'adafactor': {'value': False},
   'adam_beta1': {'value': 0.9},
   'adam_beta2': {'value': 0.999},
   'adam_epsilon': {'value': '1e-08'},
   'add_cross_attention': {'value': False},
   'architectures': {'value': None},
   'attention_probs_dropout_prob': {'value': 0.1},
   'auto_find_batch_size': {'value': False},
   'average_tokens_across_devices': {'value': False},
   'bad_words_ids': {'value': None},
   'batch_eval_metrics': {'value': False},
   'begin_suppress_tokens': {'value': None},
   'bf16': {'value': False},
   'bf16_full_eval': {'value': False},
   'bos_token_id': {'value': None},
   'chunk_size_feed_

In [5]:
list(data.keys())

["banking77[seed='42']",
 "minds14[seed='42']",
 "hwu64[seed='42']",
 "snips[seed='42']",
 "massive[seed='42']",
 "snips[seed='43']",
 "massive[seed='43']",
 "snips[seed='44']",
 "massive[seed='44']",
 "banking77[seed='43']",
 "minds14[seed='43']",
 "hwu64[seed='43']",
 "banking77[seed='44']",
 "minds14[seed='44']",
 "hwu64[seed='44']"]

In [8]:
def extract_dataset_and_seed(key):
    """
    Extract dataset name and seed from a string like "banking77[seed='44']"
    
    Args:
        key (str): String in format "dataset_name[seed='seed_value']"
        
    Returns:
        tuple: (dataset_name, seed_value)
    """
    import re
    
    # Use regex to extract dataset name and seed value
    match = re.match(r"([^[]+)\[seed='(\d+)'\]", key)
    
    if match:
        dataset_name = match.group(1)
        seed = match.group(2)
        return dataset_name, seed
    else:
        return None, None


In [10]:
from collections import defaultdict

dataset_wise_results = defaultdict(list)
for key in data.keys():
    dataset, seed = extract_dataset_and_seed(key)
    dataset_wise_results[dataset].append(data[key])

In [13]:
len(dataset_wise_results["banking77"][0])

100

- best result for each scorer
- 

In [24]:
import pandas as pd


dataset_wise_aggregated = {}
for dataset_name, dataset_results in dataset_wise_results.items():
    aggregated = []
    for seed_results in dataset_results:
        scorer_wise_results = defaultdict(list)
        for run in seed_results:
            try:
                module_name = run["module_name"].split("_")[0]
                metrics = run.get("wandb-summary", {})
                scorer_wise_results[module_name].append(metrics)
            except Exception:
                print("error processing run:", run)
                raise
        for scorer_name, scorer_runs in scorer_wise_results.items():
            best_run = max(scorer_runs, key=lambda x: x.get("scoring_accuracy", 0))
            if "scoring_accuracy" not in best_run:
                continue
            metrics = {
                "scoring_accuracy": best_run["scoring_accuracy"],
                "scoring_recall": best_run["scoring_recall"],
                "scoring_precision": best_run["scoring_precision"],
                "scoring_roc_auc": best_run["scoring_roc_auc"],
                "scoring_f1": best_run["scoring_f1"],
            }
            aggregated.append({"module_name": scorer_name, **metrics})
    dataset_wise_aggregated[dataset_name] = pd.DataFrame(aggregated)


In [29]:
dataset_wise_aggregated["banking77"].groupby("module_name").mean().sort_values(by="scoring_accuracy")

,scoring_accuracy,scoring_recall,scoring_precision,scoring_roc_auc,scoring_f1
module_name,,,,,
ptuning,0.046287,0.039310,0.008133,0.577624,0.011954
lora,0.203463,0.192324,0.146163,0.848202,0.133542
bert,0.641359,0.603224,0.621996,0.983643,0.569080
knn,0.883117,0.883482,0.895381,0.982382,0.885381
dnnc,0.888778,0.889065,0.894757,0.944127,0.888516
rerank,0.890443,0.889234,0.900021,0.977895,0.890860
sklearn,0.898102,0.895049,0.910103,0.996657,0.897871
linear,0.905095,0.905016,0.914970,0.998893,0.906807
